# Model Training and Evaluation

This notebook demonstrates the IDSNet architecture and training/evaluation functions.

Topics:
1. **Model Architecture**: Feedforward network with BatchNorm and Dropout
2. **Training Loop**: train_one_epoch() function
3. **Evaluation**: Comprehensive per-class metrics
4. **Visualization**: Confusion matrix and learning curves

In [1]:
import sys
sys.path.insert(0, '../src')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Import custom modules
from model import IDSNet, train_one_epoch, evaluate, print_evaluation_metrics

print("✓ All modules imported successfully")
print(f"✓ CUDA available: {torch.cuda.is_available()}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Using device: {device}")

/usr/lib/python3/dist-packages/pytz/__init__.py:31: SyntaxWarning: invalid escape sequence '\s'
  match = re.match("^#\s*version\s*([0-9a-z]*)\s*$", line)


✓ All modules imported successfully
✓ CUDA available: False
✓ Using device: cpu


## Test 1: Model Architecture Inspection

**Understanding IDSNet design**: The neural network for IDS classification.

**Architecture summary:**
- **Input layer**: 77 network flow features
- **Hidden layer 1**: 128 neurons + ReLU + BatchNorm + Dropout(0.3)
- **Hidden layer 2**: 64 neurons + ReLU + BatchNorm + Dropout(0.3)
- **Hidden layer 3**: 32 neurons + ReLU + BatchNorm + Dropout(0.3)
- **Output layer**: 15 neurons (attack classes) + Softmax

**Key components:**
- **ReLU**: Non-linearity, enables learning complex patterns
- **BatchNorm**: Normalizes activations, improves training stability
- **Dropout(0.3)**: Prevents overfitting by randomly disabling 30% of neurons
- **Softmax**: Converts scores to class probabilities

**Parameter count**: ~33K parameters (lightweight, suitable for federated learning)

In [2]:
# Create model
model = IDSNet(n_features=77, n_classes=15, dropout=0.3)
model.to(device)

print("=" * 70)
print("MODEL ARCHITECTURE")
print("=" * 70)
print(model)
print("\n" + "=" * 70)
print("MODEL PARAMETERS")
print("=" * 70)

total_params = 0
for name, param in model.named_parameters():
    num_params = param.numel()
    total_params += num_params
    print(f"{name:<40} {str(param.shape):<20} {num_params:>10,}")

print(f"\n{'Total Parameters':<40} {total_params:>30,}")

INFO:model:IDSNet initialized: 77 -> 128 -> 64 -> 32 -> 15


MODEL ARCHITECTURE
IDSNet(
  (fc1): Linear(in_features=77, out_features=128, bias=True)
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout1): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout2): Dropout(p=0.3, inplace=False)
  (fc3): Linear(in_features=64, out_features=32, bias=True)
  (bn3): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout3): Dropout(p=0.3, inplace=False)
  (fc4): Linear(in_features=32, out_features=15, bias=True)
  (relu): ReLU()
)

MODEL PARAMETERS
fc1.weight                               torch.Size([128, 77])      9,856
fc1.bias                                 torch.Size([128])           128
bn1.weight                               torch.Size([128])           128
bn1.bias                                 torch.Size([128])           128
f

## Test 2: Forward Pass and Predictions

**Testing inference**: Verify model produces valid predictions.

**Functions tested:**
1. **model(X)**: Raw logits (input to softmax)
   - Shape: (batch_size, 15 classes)
   - No probability normalization
   
2. **model.predict(X)**: Class predictions
   - Argmax of probabilities
   - Returns integer class index (0-14)
   
3. **model.predict_proba(X)**: Probability distribution
   - Softmax of logits
   - Sum to 1.0 per sample
   - Confidence scores

**Expected checks:**
- Output shapes correct
- Probabilities sum to ~1.0
- Max/mean probabilities reasonable
- All 15 classes can be predicted

In [3]:
print("\n" + "=" * 70)
print("FORWARD PASS TEST")
print("=" * 70)

# Create dummy data
X_test = torch.randn(5, 77).to(device)

# Forward pass
logits = model(X_test)
predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)

print(f"\nInput shape: {X_test.shape}")
print(f"Output logits shape: {logits.shape}")
print(f"Predictions shape: {predictions.shape}")
print(f"Probabilities shape: {probabilities.shape}")

print(f"\nSample predictions:")
for i in range(5):
    pred_class = predictions[i].item()
    pred_prob = probabilities[i, pred_class].item()
    print(f"  Sample {i}: Class {pred_class} (confidence: {pred_prob:.4f})")

print(f"\nProbability statistics:")
print(f"  Max probability: {probabilities.max().item():.4f}")
print(f"  Mean probability: {probabilities.mean().item():.4f}")
print(f"  Min probability: {probabilities.min().item():.4f}")

# Verify probabilities sum to 1
prob_sums = probabilities.sum(dim=1)
print(f"  Probability sums (should be ~1.0): min={prob_sums.min().item():.6f}, max={prob_sums.max().item():.6f}")


FORWARD PASS TEST

Input shape: torch.Size([5, 77])
Output logits shape: torch.Size([5, 15])
Predictions shape: torch.Size([5])
Probabilities shape: torch.Size([5, 15])

Sample predictions:
  Sample 0: Class 0 (confidence: 0.0910)
  Sample 1: Class 3 (confidence: 0.1312)
  Sample 2: Class 1 (confidence: 0.1072)
  Sample 3: Class 3 (confidence: 0.0721)
  Sample 4: Class 10 (confidence: 0.0351)

Probability statistics:
  Max probability: 0.2076
  Mean probability: 0.0667
  Min probability: 0.0127
  Probability sums (should be ~1.0): min=1.000000, max=1.000000


## Test 3: Training Loop

**End-to-end training process**: Forward pass, loss, backward pass, weight update.

**Training components:**
1. **Optimizer**: Adam (adaptive learning rates)
2. **Loss function**: Cross-entropy (suitable for multi-class classification)
3. **Batch processing**: 64 samples per update
4. **Epochs**: Full passes over training data

**train_one_epoch() function:**
- Iterates through mini-batches
- Computes predictions via forward pass
- Calculates loss
- Backpropagates gradients (backward pass)
- Updates weights (optimizer step)
- Returns average loss

**Expected behavior:**
- Loss decreases over epochs
- Demonstrates learning capability

In [4]:
print("\n" + "=" * 70)
print("TRAINING LOOP TEST")
print("=" * 70)

# Create dataset with imbalanced classes
np.random.seed(42)
n_samples = 1000
X_train = torch.randn(n_samples, 77).float()
y_train = torch.randint(0, 15, (n_samples,))  # Random class distribution

dataset = TensorDataset(X_train, y_train)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

print(f"\nDataset: {n_samples} samples, batch_size=64")
print(f"Number of batches: {len(dataloader)}")
print(f"Class distribution in training data:")
unique, counts = torch.unique(y_train, return_counts=True)
for cls, count in zip(unique, counts):
    print(f"  Class {cls}: {count} ({count/n_samples*100:.1f}%)")

# Setup training
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

print("\n" + "-" * 70)
print("Training for 3 epochs...")
print("-" * 70)

train_losses = []
for epoch in range(3):
    avg_loss = train_one_epoch(model, dataloader, optimizer, criterion, device)
    train_losses.append(avg_loss)
    print(f"Epoch {epoch+1}/3 | Loss: {avg_loss:.4f}")

print(f"\nTraining completed. Final loss: {train_losses[-1]:.4f}")


TRAINING LOOP TEST

Dataset: 1000 samples, batch_size=64
Number of batches: 16
Class distribution in training data:
  Class 0: 63 (6.3%)
  Class 1: 70 (7.0%)
  Class 2: 67 (6.7%)
  Class 3: 67 (6.7%)
  Class 4: 56 (5.6%)
  Class 5: 60 (6.0%)
  Class 6: 50 (5.0%)
  Class 7: 77 (7.7%)
  Class 8: 62 (6.2%)
  Class 9: 61 (6.1%)
  Class 10: 69 (6.9%)
  Class 11: 75 (7.5%)
  Class 12: 74 (7.4%)
  Class 13: 69 (6.9%)
  Class 14: 80 (8.0%)


INFO:model:Epoch completed | Average Loss: 2.7876
INFO:model:Epoch completed | Average Loss: 2.7478
INFO:model:Epoch completed | Average Loss: 2.7207



----------------------------------------------------------------------
Training for 3 epochs...
----------------------------------------------------------------------
Epoch 1/3 | Loss: 2.7876
Epoch 2/3 | Loss: 2.7478
Epoch 3/3 | Loss: 2.7207

Training completed. Final loss: 2.7207


In [ ]:
# Plot training loss
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(train_losses) + 1), train_losses, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Epoch')
plt.ylabel('Average Loss')
plt.title('Training Loss Over Epochs')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/plots/training_loss.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Training loss plot saved")

## Test 4: Evaluation and Metrics

**Comprehensive performance assessment**: Multi-class classification metrics.

**Metrics computed:**
- **Accuracy**: Overall correctness (can mislead on imbalanced data)
- **Macro metrics**: Equal weight to each class (good for imbalanced data)
- **Weighted metrics**: Weight by class frequency (overall performance)
- **Per-class results**: Individual class detection rates

**For 15-class IDS:**
- Macro-F1 important (not all attacks are equally frequent)
- Per-class insights reveal which attacks are hard to detect
- Confusion matrix shows which classes are confused with each other

**Expected output:**
- Confusion matrix: 15×15 heatmap
- Per-class precision/recall/F1
- Overall accuracy and macro-F1

In [5]:
print("\n" + "=" * 70)
print("EVALUATION TEST")
print("=" * 70)

# Create evaluation dataset
X_eval = torch.randn(500, 77).float()
y_eval = torch.randint(0, 15, (500,))

eval_dataset = TensorDataset(X_eval, y_eval)
eval_dataloader = DataLoader(eval_dataset, batch_size=64)

print(f"\nEvaluation dataset: {len(X_eval)} samples")
print(f"Batch size: 64, Number of batches: {len(eval_dataloader)}")

# Run evaluation
results = evaluate(model, eval_dataloader, device)

print(f"\nEvaluation Results:")
print(f"  Accuracy: {results['accuracy']:.4f}")
print(f"  Macro Precision: {results['macro_precision']:.4f}")
print(f"  Macro Recall: {results['macro_recall']:.4f}")
print(f"  Macro F1: {results['macro_f1']:.4f}")
print(f"  Weighted Precision: {results['weighted_precision']:.4f}")
print(f"  Weighted Recall: {results['weighted_recall']:.4f}")
print(f"  Weighted F1: {results['weighted_f1']:.4f}")

INFO:model:Evaluation Results:
INFO:model:  Accuracy: 0.0720
INFO:model:  Macro F1-Score: 0.0543
INFO:model:  Weighted F1-Score: 0.0547
INFO:model:  Samples evaluated: 500



EVALUATION TEST

Evaluation dataset: 500 samples
Batch size: 64, Number of batches: 8

Evaluation Results:
  Accuracy: 0.0720
  Macro Precision: 0.0629
  Macro Recall: 0.0727
  Macro F1: 0.0543
  Weighted Precision: 0.0659
  Weighted Recall: 0.0720
  Weighted F1: 0.0547


In [6]:
# Print detailed metrics
print_evaluation_metrics(results)


EVALUATION METRICS

Overall Performance:
  Accuracy:          0.0720
  Macro Precision:   0.0629
  Macro Recall:      0.0727
  Macro F1-Score:    0.0543
  Weighted Precision: 0.0659
  Weighted Recall:   0.0720
  Weighted F1-Score: 0.0547

Per-Class Metrics:
Class                          Precision    Recall       F1-Score     Support 
--------------------------------------------------------------------------
Class 0                        0.0345       0.0400       0.0370       25      
Class 1                        0.1154       0.3158       0.1690       38      
Class 2                        0.2000       0.0256       0.0455       39      
Class 3                        0.0323       0.0625       0.0426       32      
Class 4                        0.0500       0.0213       0.0299       47      
Class 5                        0.0625       0.1000       0.0769       30      
Class 6                        0.0000       0.0000       0.0000       38      
Class 7                        0.1

## Test 5: Confusion Matrix Visualization

**Prediction error analysis**: Which predictions are confused?

**Interpretation guide:**
- **Diagonal (blue)**: Correct predictions
- **Off-diagonal**: Misclassifications
- **Darker blue**: More samples
- **Row**: True class
- **Column**: Predicted class

**What to look for:**
- High diagonal: Good model
- Clusters off-diagonal: Classes confused together (related attacks?)
- Specific rows/cols problematic: Certain attacks hard to detect or often misidentified

**Example:** If DDoS (class 0) often predicted as NetScan (class 3), they may have similar flow patterns

In [ ]:
# Plot confusion matrix
conf_matrix = results['confusion_matrix']

plt.figure(figsize=(12, 10))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar_kws={'label': 'Count'})
plt.xlabel('Predicted Class')
plt.ylabel('True Class')
plt.title('Confusion Matrix (15 classes)')
plt.tight_layout()
plt.savefig('../results/plots/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Confusion matrix plot saved")

## Test 6: Per-Class Performance

**Fine-grained attack detection analysis**: How well does each attack class perform?

**Metrics per attack class:**
- **Precision**: % predicted as this attack that are actually this attack
  - High precision: Few false positives
  - Low precision: Often misidentify other attacks as this one
  
- **Recall**: % actual attacks of this type that are correctly identified
  - High recall: Catch most instances of this attack
  - Low recall: Miss many instances (dangerous for IDS!)
  
- **F1-Score**: Balance between precision and recall
  - Important for imbalanced attacks

- **Support**: Number of test samples for this class

**Patterns to notice:**
- Rare attack types (low support): May have lower F1
- Specific attack types: Consistently bad F1 suggests they're hard to distinguish
- Trade-off: Some classes high precision but low recall (conservative)

In [ ]:
# Extract per-class metrics
classes = sorted(results['per_class'].keys())
precisions = [results['per_class'][c]['precision'] for c in classes]
recalls = [results['per_class'][c]['recall'] for c in classes]
f1_scores = [results['per_class'][c]['f1'] for c in classes]
supports = [results['per_class'][c]['support'] for c in classes]

# Plot per-class F1 scores
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# F1 scores by class
colors = plt.cm.viridis(np.linspace(0, 1, len(classes)))
bars = ax1.bar(classes, f1_scores, color=colors)
ax1.set_xlabel('Class')
ax1.set_ylabel('F1-Score')
ax1.set_title('Per-Class F1-Scores')
ax1.set_ylim([0, 1])
ax1.grid(True, alpha=0.3, axis='y')

# Support (number of samples) by class
ax2.bar(classes, supports, color=colors)
ax2.set_xlabel('Class')
ax2.set_ylabel('Support (number of samples)')
ax2.set_title('Class Support in Evaluation Set')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../results/plots/per_class_performance.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Per-class performance plot saved")

In [ ]:
# Precision-Recall curve data (useful for multi-class)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Precision vs Recall scatter
ax1.scatter(recalls, precisions, s=100, alpha=0.6, c=classes, cmap='viridis')
for i, cls in enumerate(classes):
    ax1.annotate(f'C{cls}', (recalls[i], precisions[i]), fontsize=8, ha='center')
ax1.set_xlabel('Recall')
ax1.set_ylabel('Precision')
ax1.set_title('Precision vs Recall by Class')
ax1.set_xlim([0, 1])
ax1.set_ylim([0, 1])
ax1.grid(True, alpha=0.3)
ax1.plot([0, 1], [1, 0], 'k--', alpha=0.3, label='Random')

# Macro vs Weighted metrics comparison
metrics_names = ['Precision', 'Recall', 'F1']
macro_values = [results['macro_precision'], results['macro_recall'], results['macro_f1']]
weighted_values = [results['weighted_precision'], results['weighted_recall'], results['weighted_f1']]

x = np.arange(len(metrics_names))
width = 0.35

ax2.bar(x - width/2, macro_values, width, label='Macro', alpha=0.8)
ax2.bar(x + width/2, weighted_values, width, label='Weighted', alpha=0.8)
ax2.set_ylabel('Score')
ax2.set_title('Macro vs Weighted Averages')
ax2.set_xticks(x)
ax2.set_xticklabels(metrics_names)
ax2.set_ylim([0, 1])
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../results/plots/metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Metrics comparison plot saved")

In [ ]:
print("\n" + "=" * 70)
print("✓ MODEL TRAINING NOTEBOOK COMPLETED")
print("=" * 70)
print(f"\nSummary:")
print(f"  Model: IDSNet (77 -> 128 -> 64 -> 32 -> 15)")
print(f"  Total Parameters: {total_params:,}")
print(f"  Training Loss: {train_losses[-1]:.4f}")
print(f"  Evaluation Accuracy: {results['accuracy']:.4f}")
print(f"  Evaluation Macro F1: {results['macro_f1']:.4f}")
print(f"\nGenerated Plots:")
print(f"  - results/plots/training_loss.png")
print(f"  - results/plots/confusion_matrix.png")
print(f"  - results/plots/per_class_performance.png")
print(f"  - results/plots/metrics_comparison.png")